# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/heyzara124-hub/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Question:** Among a client's visible search pages, which ones are earning noticeably fewer
clicks than similar pages at the same search position — and can a model rank those pages more
accurately than a simple hand-written rule?

**Decision this supports:** a FlyRank content reviewer has limited time each week and needs to
know which pages to open first. This work turns "check everything eventually" into a ranked,
reasoned queue: check these pages first, and here's why each one made the list.

**Lane:** CTR / Engagement Opportunity Scoring.

In [8]:
print('Lane: CTR / Engagement Opportunity Scoring')
print('Unit of analysis: one content page (content_id)')
print('Output: a ranked queue with a reason code and action per page')

Lane: CTR / Engagement Opportunity Scoring
Unit of analysis: one content page (content_id)
Output: a ranked queue with a reason code and action per page


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Release:** the anonymized starter dataset shipped in this repo,
`data/raw/content_refresh_anonymized.csv` — 30,000 content pages across 32 pseudonymized
clients, a single trailing-90-day snapshot.

**Filter applied:** I only score pages with `impressions_90d ≥ 500` and an `avg_position`
between 1 and 20 — the "volume floor" from Week 2. Below that floor, a CTR comparison is mostly
noise (a page with 10 impressions can swing from 0% to 30% CTR on one click). This filter keeps
12,023 of the 30,000 pages.

**Columns deliberately excluded as model features:** `ctr` and `clicks_90d` (the label's own
ingredients — using them would be circular), `trend_direction` and `trend_pct` (the source of
`is_declining_label`, banned per the data dictionary), and `content_id` / `client_id`
(identifiers — used only for grouping and splits, never as inputs).

No client names, domains, URLs, or raw exports appear anywhere in this notebook or its outputs.

In [9]:
import pandas as pd
import numpy as np
import os

if not os.path.exists('data/raw/content_refresh_anonymized.csv'):
    if not os.path.exists('flyrank-ml-internship'):
        !git clone -q https://github.com/heyzara124-hub/flyrank-ml-internship.git
    os.chdir('flyrank-ml-internship')

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
visible = df[(df['impressions_90d'] >= 500) & (df['avg_position'] > 0) & (df['avg_position'] <= 20)].copy()
print(f'Full dataset: {len(df):,} pages, {df["client_id"].nunique()} clients')
print(f'After volume floor: {len(visible):,} pages, {visible["client_id"].nunique()} clients')

Full dataset: 30,000 pages, 32 clients
After volume floor: 12,023 pages, 28 clients


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Target (proxy):** `ctr_gap` = a page's own `ctr` minus the median `ctr` of other visible
pages at the same `position_tier`. Built entirely from observed clicks and impressions — not a
hand-defined rule — so it satisfies the "target must be observed" bar from the framing skill.

**Baseline (Week 4):** a transparent rule — flag a page if its CTR is under half its position
tier's median CTR, and it has at least 500 impressions. Rank flagged pages by traffic volume.

**Model (Week 5):** a Random Forest Regressor predicting `ctr_gap` from content and traffic
signals (search volume, competition, word/char count, content age, freshness, engagement rate,
scroll rate, AI traffic share, content type, intent, position tier).

**Validation design:** client-holdout split (`GroupShuffleSplit` on `client_id`, 80/20). No
client's pages appear in both train and test — this matters because pages from the same client
often share templates and content strategy, so a random row split would let the model partly
memorize client style instead of learning a generalizable pattern.

**Leakage check (Week 6):** confirmed the final feature list contains none of `ctr`,
`clicks_90d`, `tier_median_ctr`, `trend_direction`, `trend_pct`. Also compared the honest
client-holdout split against a leaky random-row split: R2 was inflated under the leaky split
(0.43 vs 0.36), the expected direction if leakage were present, which is why the client-holdout
number is the one this paper reports.

In [10]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor

tier_median_ctr = visible.groupby('position_tier')['ctr'].median()
visible['tier_median_ctr'] = visible['position_tier'].map(tier_median_ctr)
visible['ctr_gap'] = visible['ctr'] - visible['tier_median_ctr']

numeric_features = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'content_age_days', 'days_since_last_update', 'sessions_90d',
    'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impressions_90d',
]
categorical_features = [
    'content_type', 'main_intent', 'competition_level',
    'freshness_tier', 'position_tier',
]

model_df = visible.copy()
for c in numeric_features:
    model_df[c] = model_df[c].fillna(0)
for c in categorical_features:
    model_df[c] = model_df[c].fillna('unknown')
model_df['log_impressions_90d'] = np.log1p(model_df['impressions_90d'])
numeric_features = [c if c != 'impressions_90d' else 'log_impressions_90d' for c in numeric_features]

X = model_df[numeric_features + categorical_features]
y = model_df['ctr_gap']
groups = model_df['client_id']

banned = {'ctr', 'clicks_90d', 'tier_median_ctr', 'trend_direction', 'trend_pct', 'content_id', 'client_id'}
touched = (set(numeric_features) | set(categorical_features)) & banned
print('Leakage check -- banned columns in final feature set:', touched if touched else 'none')

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
print(f'Train: {len(train_idx):,} rows | Test: {len(test_idx):,} rows | Shared clients: {len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]))}')

Leakage check -- banned columns in final feature set: none
Train: 11,202 rows | Test: 821 rows | Shared clients: 0


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

Both the baseline rule and the model are evaluated on the same held-out test clients, with
**Precision@50**: of the 50 pages each method ranks worst, how many are actually in the true
worst-50 by `ctr_gap` in the test set.

In [11]:
from sklearn.metrics import r2_score
from scipy.stats import spearmanr

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
test_df = model_df.iloc[test_idx]

pre = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)], remainder='passthrough')
model = Pipeline([('pre', pre), ('rf', RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1))])
model.fit(X_train, y_train)
pred = model.predict(X_test)

result = test_df.copy()
result['predicted_ctr_gap'] = pred
actual_top50 = set(result.sort_values('ctr_gap').head(50)['content_id'])
model_top50 = set(result.sort_values('predicted_ctr_gap').head(50)['content_id'])
model_p50 = len(actual_top50 & model_top50) / 50

baseline_flag = result['ctr'] < 0.5 * result['tier_median_ctr']
baseline_pool = result[baseline_flag].copy()
baseline_pool['score'] = baseline_pool['impressions_90d']
baseline_top50 = set(baseline_pool.sort_values('score', ascending=False).head(50)['content_id'])
baseline_p50 = len(actual_top50 & baseline_top50) / 50

r2 = r2_score(y_test, pred)
rho, pval = spearmanr(y_test, pred)

results_table = pd.DataFrame({
    'Precision@50': [baseline_p50, model_p50],
    'R2 (gap prediction)': [None, round(r2, 3)],
    'Spearman rho': [None, round(rho, 3)],
}, index=['Week-4 baseline rule', 'Random Forest model'])
print(results_table)

                      Precision@50  R2 (gap prediction)  Spearman rho
Week-4 baseline rule          0.16                  NaN           NaN
Random Forest model           0.34                0.363         0.763


## 5. Limitations

*What this work cannot claim.*

- **Single snapshot.** One 90-day window, no time-based validation yet — I don't know if this
  holds across seasons or over time. That requires the full warehouse release.
- **Proxy, not ground truth.** `ctr_gap` measures underperformance relative to peers today, not
  whether fixing a page will actually raise its CTR tomorrow. That's a claim about the future
  this snapshot can't test.
- **Correlational.** The model finds association between page traits and `ctr_gap`, not cause
  and effect. A low CTR could reflect a fixable title, or a genuine intent mismatch — this
  model doesn't distinguish those.
- **Small held-out test set.** 821 test-set rows across a handful of clients after grouping;
  Precision@50 on that few pages is noisier than a big test set would be.
- **Volume floor excludes most of the dataset.** Only 12,023 of 30,000 pages clear
  `impressions_90d ≥ 500`; this queue says nothing about the other ~18,000 lower-traffic pages.
- **No causal claim about Google's ranking algorithm anywhere in this work.**

Every number in this paper should be read as **observed, directional, decision-support** —
useful for prioritizing a human's next look, not a guarantee.

In [12]:
print('No computation for this section -- see limitations above.')

No computation for this section -- see limitations above.


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

Reusing the Week-7 playbook: score every visible page with the final model (trained on all
visible data this time, not just the train split), then tag each flagged page with a reason
code so a reviewer knows *why* it's on the list.

In [13]:
final_model = Pipeline([('pre', pre), ('rf', RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1))])
final_model.fit(X, y)
model_df['predicted_ctr_gap'] = final_model.predict(X)

freshness_median_days = model_df['days_since_last_update'].median()
model_df['wc_med_for_tier'] = model_df.groupby('position_tier')['word_count'].transform('median')

def reason_for(row):
    if row['days_since_last_update'] > freshness_median_days * 1.5:
        return 'stale_content_review', 'refresh_and_republish'
    elif row['word_count'] > 0 and row['word_count'] < row['wc_med_for_tier'] * 0.6:
        return 'thin_content_for_tier', 'expand_content_depth'
    else:
        return 'ctr_below_position_norm', 'review_meta_title_snippet'

reasons = model_df.apply(reason_for, axis=1)
model_df['reason_code'] = [r[0] for r in reasons]
model_df['action_label'] = [r[1] for r in reasons]

final_queue = model_df.sort_values('predicted_ctr_gap').reset_index(drop=True)
top10 = final_queue.head(10)
print(f'{len(final_queue):,} pages ranked. Top 10:\n')
print(top10[['content_id', 'position_tier', 'predicted_ctr_gap', 'reason_code', 'action_label']].to_string(index=False))
print('\nAction mix, top 50:')
print(final_queue.head(50)['action_label'].value_counts())

12,023 pages ranked. Top 10:

          content_id position_tier  predicted_ctr_gap             reason_code              action_label
content_c82bc0c24241        page_1          -0.210682 ctr_below_position_norm review_meta_title_snippet
content_c6999f7eb5fb        page_1          -0.210237 ctr_below_position_norm review_meta_title_snippet
content_ca17a024f90c        page_1          -0.209025 ctr_below_position_norm review_meta_title_snippet
content_f986bd514b6e        page_1          -0.208212 ctr_below_position_norm review_meta_title_snippet
content_72fdb385e810        page_1          -0.207411    stale_content_review     refresh_and_republish
content_f6ae0f36d70d        page_1          -0.207378 ctr_below_position_norm review_meta_title_snippet
content_11a4f985f14d        page_1          -0.206194 ctr_below_position_norm review_meta_title_snippet
content_c8e9d6ab9013        page_1          -0.205476    stale_content_review     refresh_and_republish
content_89b8af9c2e3d        page_1

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

Saving three small, public-safe artifacts to `work/outputs/` for the deployed paper: a
model-vs-baseline chart, a feature-importance chart, and an action-mix chart. All are aggregate
charts — no row-level or client-identifying data in any of them.

In [14]:
import matplotlib.pyplot as plt
import json as json_lib

os.makedirs('work/outputs/figures', exist_ok=True)

# 1. model vs baseline
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(['Baseline rule', 'Random Forest'], [baseline_p50, model_p50], color=['#b0b0b0', '#5b4fc9'])
ax.set_ylabel('Precision@50 (held-out clients)')
ax.set_title('Model vs baseline')
plt.tight_layout()
plt.savefig('work/outputs/figures/model_vs_baseline.svg')
plt.close()

# 2. feature importance
ohe = final_model.named_steps['pre'].named_transformers_['cat']
cat_names = list(ohe.get_feature_names_out(categorical_features))
remainder_names = [c for c in X.columns if c not in categorical_features]
all_names = cat_names + remainder_names
importances = pd.Series(final_model.named_steps['rf'].feature_importances_, index=all_names).sort_values(ascending=False).head(8)

fig, ax = plt.subplots(figsize=(7, 4))
importances.sort_values().plot(kind='barh', ax=ax, color='#5b4fc9')
ax.set_xlabel('Feature importance')
ax.set_title('What the model leans on most')
plt.tight_layout()
plt.savefig('work/outputs/figures/feature_importance.svg')
plt.close()

# 3. action mix
fig, ax = plt.subplots(figsize=(5, 4))
final_queue.head(50)['action_label'].value_counts().plot(kind='bar', ax=ax, color='#5b4fc9')
ax.set_ylabel('Pages in top 50')
ax.set_title('Recommended actions, top 50')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig('work/outputs/figures/action_mix.svg')
plt.close()

# metrics JSON -- small, committed, the receipts the paper's numbers trace back to
capstone_metrics = {
    'queue_size': int(len(final_queue)),
    'model_precision_at_50': round(model_p50, 3),
    'baseline_precision_at_50': round(baseline_p50, 3),
    'r2': round(float(r2), 3),
    'spearman_rho': round(float(rho), 3),
    'top50_action_mix': final_queue.head(50)['action_label'].value_counts().to_dict(),
}
with open('work/outputs/capstone_metrics.json', 'w') as f:
    json_lib.dump(capstone_metrics, f, indent=2)

print('Saved 3 charts to work/outputs/figures/ and work/outputs/capstone_metrics.json')
print(capstone_metrics)

Saved 3 charts to work/outputs/figures/ and work/outputs/capstone_metrics.json
{'queue_size': 12023, 'model_precision_at_50': 0.34, 'baseline_precision_at_50': 0.16, 'r2': 0.363, 'spearman_rho': 0.763, 'top50_action_mix': {'review_meta_title_snippet': 38, 'refresh_and_republish': 11, 'expand_content_depth': 1}}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.

---

## ML-12 — Closing: demo outline, social cut, employer summary

**5-minute demo outline** (what I'd show live):
1. *(30s)* The problem: a content reviewer can't check thousands of pages by hand — show the
   scale (12,023 scoreable pages).
2. *(60s)* The baseline: my Week-4 hand-written rule, and why it's a fair starting point.
3. *(90s)* The model: show the Precision@50 table — baseline 0.16, model 0.34 — and explain the
   client-holdout split in one sentence.
4. *(60s)* The queue: walk through 2–3 rows from the top 10, reading the reason code out loud.
5. *(60s)* Limitations, said out loud, not buried: single snapshot, proxy target, decision
   support only.

**Social-post cut** (short, for LinkedIn):
> Built a content-scoring model for my FlyRank ML internship capstone: it ranks underperforming
> pages by how far their click-through rate sits below similar pages at the same search
> position. On a held-out test, it roughly doubled a hand-written baseline rule's precision at
> flagging the right pages. Full write-up and code linked below.

**Employer-facing summary** (3 sentences):
I built and validated a machine learning model that ranks web pages by how much they
underperform expected click-through rates for their search position, using a client-holdout
split to avoid data leakage. The model more than doubled a transparent baseline rule's accuracy
at flagging the right pages to review first (Precision@50: 0.34 vs 0.16). The full pipeline —
data contract, baseline, model, leakage audit, and action playbook — is documented and
reproducible end to end in my public GitHub repo.